In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Load and preprocess the data
df = pd.read_excel(r'../concrete data.xlsx', sheet_name='fc')
df.dropna(inplace=True)

Y = df.iloc[:, 0].values
X = df.iloc[:, 1:].values

# Normalize the features
x_mean = X.mean(0)
x_std = X.std(0)
X_normal = (X - x_mean) / x_std

y_mean = Y.mean()
y_std = Y.std()
Y_normal = (Y - y_mean) / y_std

Xtrain, Xtest, ytrain, ytest = train_test_split(X_normal, Y_normal, train_size=0.85, random_state=42)

Xtrain = torch.tensor(Xtrain, dtype=torch.float32)
Xtest = torch.tensor(Xtest, dtype=torch.float32)
ytrain = torch.tensor(ytrain, dtype=torch.float32).unsqueeze(1)
ytest = torch.tensor(ytest, dtype=torch.float32).unsqueeze(1)

# Autoencoder Model for Feature Selection
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed, latent

# Initialize the Autoencoder
latent_dim = 5  # Adjust based on the desired level of dimensionality reduction
autoencoder = Autoencoder(Xtrain.shape[1], latent_dim)
optimizer_ae = optim.Adam(autoencoder.parameters(), lr=0.001)
criterion_ae = nn.MSELoss()

# Train the Autoencoder
num_epochs = 100
batch_size = 8

for epoch in range(num_epochs):
    autoencoder.train()
    for i in range(0, len(Xtrain), batch_size):
        batch_X = Xtrain[i:i + batch_size]
        
        optimizer_ae.zero_grad()
        reconstructed, _ = autoencoder(batch_X)
        loss = criterion_ae(reconstructed, batch_X)
        loss.backward()
        optimizer_ae.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item()}')

# Extract the latent features from the trained Autoencoder
autoencoder.eval()
with torch.no_grad():
    _, Xtrain_latent = autoencoder(Xtrain)
    _, Xtest_latent = autoencoder(Xtest)

# Regression Model using Latent Features
class RegressionModel(nn.Module):
    def __init__(self, input_dim, layers, neurons):
        super(RegressionModel, self).__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, neurons))
        self.layers.append(nn.ReLU())
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(neurons, neurons))
            self.layers.append(nn.ReLU())
        self.layers.append(nn.Linear(neurons, 1))
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Initialize and Train the Regression Model
best_layers = 2
best_neurons = 53
best_learn_rate = 0.0014953325321920149

model = RegressionModel(Xtrain_latent.shape[1], best_layers, best_neurons)
optimizer = optim.Adam(model.parameters(), lr=best_learn_rate)
criterion = nn.MSELoss()

# Train the Regression Model
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(Xtrain_latent)
    loss = criterion(outputs, ytrain)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item()}')

# Evaluate the Model
model.eval()
with torch.no_grad():
    ytrain_pred = model(Xtrain_latent)
    ytest_pred = model(Xtest_latent)

ytrain_pred = ytrain_pred.numpy()
ytest_pred = ytest_pred.numpy()

train_r2 = r2_score(ytrain.numpy(), ytrain_pred)
test_r2 = r2_score(ytest.numpy(), ytest_pred)

print(f'Train R-squared: {train_r2}')
print(f'Test R-squared: {test_r2}')

R2 = r2_score(ytest.numpy(), ytest_pred)
RMSE = mean_squared_error(ytest.numpy(), ytest_pred) ** 0.5
MAE = mean_absolute_error(ytest.numpy(), ytest_pred)
print(f'R-squared is {R2}, RMSE is {RMSE}, and MAE is {MAE}.')


Epoch [10/100], Loss: 0.521198034286499
Epoch [20/100], Loss: 0.49985191226005554
Epoch [30/100], Loss: 0.4978867769241333
Epoch [40/100], Loss: 0.4863261580467224
Epoch [50/100], Loss: 0.48407819867134094
Epoch [60/100], Loss: 0.4811897277832031
Epoch [70/100], Loss: 0.48562711477279663
Epoch [80/100], Loss: 0.4836411774158478
Epoch [90/100], Loss: 0.4781210124492645
Epoch [100/100], Loss: 0.47768303751945496
Epoch [10/100], Loss: 0.7805949449539185
Epoch [20/100], Loss: 0.6946402192115784
Epoch [30/100], Loss: 0.6291362643241882
Epoch [40/100], Loss: 0.589438259601593
Epoch [50/100], Loss: 0.5613300800323486
Epoch [60/100], Loss: 0.5407620668411255
Epoch [70/100], Loss: 0.5232420563697815
Epoch [80/100], Loss: 0.5075728297233582
Epoch [90/100], Loss: 0.49334093928337097
Epoch [100/100], Loss: 0.4802429676055908
Train R-squared: 0.5181314667692514
Test R-squared: 0.45988410585101247
R-squared is 0.45988410585101247, RMSE is 0.7470379990713857, and MAE is 0.561981201171875.
